In [1]:
# ── Install (run once, then restart kernel) ──────────────────────────────────
# On a fresh GCP TPU VM torch_xla may already be present; this is a no-op then.
!pip install -q torch==2.9.0 torch_xla[tpu]==2.9.0 \
    -f https://storage.googleapis.com/libtpu-releases/index.html \
    --no-warn-script-location

!pip install -q tiktoken "datasets>=3.0" tqdm

print("Done. Restart the kernel now, then run cells.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.8.0+cpu requires torch==2.8.0, but you have torch 2.9.0 which is incompatible.
torchvision 0.23.0+cpu requires torch==2.8.0, but you have torch 2.9.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Done. Restart the kernel now, then run cells.


In [2]:
import os, math, time, json, threading, queue
from typing import Optional, Iterator
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from datasets import load_dataset

In [3]:
import os, math, time, json, threading, queue
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
print("torch:", torch.__version__)

torch: 2.9.0+cu128


In [4]:
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
print("torch_xla:", torch_xla.__version__)

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:246: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


torch_xla: 2.9.0


In [5]:
os.environ['PJRT_DEVICE'] = 'TPU'
print(xr.global_runtime_device_count())

E0000 00:00:1773150497.749690      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238


8


In [6]:
import torch_xla.distributed.spmd as xs
xr.use_spmd()
print("SPMD ok")

SPMD ok


In [7]:
import numpy as np
NUM_DEVICES = xr.global_runtime_device_count()
print("devices:", NUM_DEVICES)

import torch_xla.distributed.spmd as xs
MESH = xs.Mesh(np.arange(NUM_DEVICES), (NUM_DEVICES,), ('data',))
print("Mesh:", MESH)

devices: 8
Mesh: {'device_ids': [0, 1, 2, 3, 4, 5, 6, 7], 'mesh_shape': (8,), 'axis_names': ('data',)}


In [8]:
# ═══════════════════════════════════════════════════════════════
# MODEL HYPERPARAMETERS
# nanochat 'depth' dial → all other dims derived automatically
# ═══════════════════════════════════════════════════════════════
DEPTH      = 12       # nanochat depth dial (12 → GPT-2 class)
ASPECT     = 64       # nanochat constant: n_embd = depth × aspect
HEAD_DIM   = 64       # fixed head dim (nanochat)

N_LAYER    = DEPTH
N_EMBD     = DEPTH * ASPECT          # 768
N_HEAD     = N_EMBD  // HEAD_DIM     # 12
N_KV_HEAD  = N_HEAD                  # 12  (no KV compression at this scale)
SEQ_LEN    = 1024                    # context window

# Vocab: tiktoken r50k_base (GPT-2) = 50 257 tokens
# Padded to nearest multiple of 128 for TPU tensor-core efficiency
VOCAB_RAW   = 50_257
VOCAB_SIZE  = ((VOCAB_RAW + 127) // 128) * 128   # 50 304

# ═══════════════════════════════════════════════════════════════
# TRAINING BUDGET  (Chinchilla-optimal: 20 × compute-params)
# ═══════════════════════════════════════════════════════════════
TARGET_COMPUTE_PARAMS = 124_000_000   # GPT-2 base reference
TARGET_TOKENS         = 20 * TARGET_COMPUTE_PARAMS   # 2.48 B

# ── Batch ──────────────────────────────────────────────────────
# Kept at 2^17 (16 seqs/chip × 8 chips × 1024 tokens).
#
# WHY NOT 2^18: doubling the batch doubles activations per chip:
#   • bf16[32,1024,50304] logit tensors → 6.1 GB
#   • f32[384,1024,1024]  SDPA scores × 12 layers → 18 GB
#   Both together exceed the 15.75 GB v5e HBM budget.
#
# MFU is instead recovered via:
#   1. Chunked cross-entropy (avoids ever materialising the full logit
#      buffer; saves ~1.5 GB peak HBM even at 2^17, less fragmentation)
#   2. Bulk-extend data pipeline (eliminates CPU→TPU data stalls)
#   3. .reshape() instead of .contiguous().view() in attention
#   4. LOG_EVERY=50  (fewer host-sync round-trips per 1000 steps)
TOTAL_BATCH_TOKENS = 2 ** 18   # 262 144  (32 seqs/chip × 8 chips × 1024)
TOTAL_SEQS         = TOTAL_BATCH_TOKENS // SEQ_LEN   # 256
PER_CHIP_SEQS      = TOTAL_SEQS // NUM_DEVICES        # 32
NUM_STEPS          = math.ceil(TARGET_TOKENS / TOTAL_BATCH_TOKENS)  # ~18 920

# ── LR: nanochat defaults, scaled by (n_embd/768)^-0.5 ────────
LR_SCALE      = (N_EMBD / 768) ** -0.5    # = 1.0 at depth 12
MATRIX_LR     = 0.02                       # Muon
EMBED_LR      = 0.200 * LR_SCALE           # AdamW for wte
LMHEAD_LR     = 0.004 * LR_SCALE           # AdamW for lm_head
SCALAR_LR     = 0.005                      # AdamW for resid/x0 lambdas
X0_LR         = 0.5                        # AdamW for x0_lambdas (higher)

ADAM_BETAS    = (0.8, 0.95)
ADAM_EPS      = 1e-10
MUON_MOMENTUM = 0.95
MUON_NS_STEPS = 5

# Weight decay scales ∝ 1/depth² (nanochat: tuned at d12, = 0.2)
WEIGHT_DECAY  = 0.2 * (12 / DEPTH) ** 2    # = 0.2 at depth 12

# ── LR schedule: no warmup, 40 % linear warmdown to 0 ──────────
WARMDOWN_START = int(NUM_STEPS * 0.6)   # begin warmdown at 60 %

# ── Logging ────────────────────────────────────────────────────
# LOG_EVERY=50: each loss.item() forces a host sync; 50-step cadence
# reduces blocking overhead vs the original 20-step cadence.
LOG_EVERY  = 100
VAL_EVERY  = 200

# ── Prompt evaluation ──────────────────────────────────────────
SAMPLE_EVAL_EVERY = 3_000
EVAL_PROMPTS = [
    "The capital of France is",
    "In 1969, humans first",
    "The chemical formula for water is",
    "Machine learning is a field of",
    "Once upon a time, in a land far away,",
    "The theory of relativity states that",
    "To train a neural network, you need",
    "The largest planet in our solar system is",
]

CKPT_EVERY = 5000
CKPT_DIR   = './checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# CE along T-axis: 8 sequential [B, T//8, V] matmuls instead of one [B,T,V]
# — avoids materialising the 26GB full logit buffer at 2^18 batch.
CE_CHUNKS = 8   # SEQ_LEN (1024) must be divisible by this

USE_WANDB  = False   # flip to True to enable wandb logging

print(f'n_layer={N_LAYER}  n_embd={N_EMBD}  n_head={N_HEAD}')
print(f'seq_len={SEQ_LEN}  vocab={VOCAB_SIZE}')
print(f'steps={NUM_STEPS:,}  batch={TOTAL_BATCH_TOKENS:,} tokens  per_chip={PER_CHIP_SEQS} seqs')
print(f'Training tokens: {TARGET_TOKENS/1e9:.2f} B')
print(f'Prompt eval every {SAMPLE_EVAL_EVERY} steps ({len(EVAL_PROMPTS)} prompts)')


n_layer=12  n_embd=768  n_head=12
seq_len=1024  vocab=50304
steps=18,921  batch=131,072 tokens  per_chip=16 seqs
Training tokens: 2.48 B
Prompt eval every 3000 steps (8 prompts)


In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  nanochat GPT Architecture  —  faithfully ported to XLA      ║
# ╚══════════════════════════════════════════════════════════════╝

from typing import Optional

# ── Utility: RMSNorm (no learnable weight, nanochat style) ──────────────────
def rms_norm(x: torch.Tensor) -> torch.Tensor:
    """Parameter-free RMSNorm using F.rms_norm (PyTorch ≥ 2.4)."""
    return F.rms_norm(x, (x.size(-1),))


# ── Rotary Positional Embeddings (RoPE) ─────────────────────────────────────
def precompute_rotary(seq_len: int, head_dim: int,
                      base: float = 10_000.0) -> tuple[torch.Tensor, torch.Tensor]:
    """Returns (cos, sin) buffers, shape (1, seq_len, 1, head_dim//2)."""
    half  = head_dim // 2
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t     = torch.arange(seq_len, dtype=torch.float32)
    freqs = torch.outer(t, theta)
    cos   = freqs.cos()[None, :, None, :]   # (1, T, 1, half)
    sin   = freqs.sin()[None, :, None, :]
    return cos.to(torch.bfloat16), sin.to(torch.bfloat16)


def apply_rotary(x: torch.Tensor,
                 cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """x: (B, T, H, D);  cos/sin: (1, T, 1, D//2)"""
    half = x.size(-1) // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


# ── Causal Self-Attention (GQA-ready, SDPA backend) ─────────────────────────
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, n_kv_head: int, head_dim: int):
        super().__init__()
        self.n_head    = n_head
        self.n_kv_head = n_kv_head
        self.head_dim  = head_dim
        self.n_rep     = n_head // n_kv_head

        self.c_q    = nn.Linear(n_embd, n_head    * head_dim, bias=False)
        self.c_k    = nn.Linear(n_embd, n_kv_head * head_dim, bias=False)
        self.c_v    = nn.Linear(n_embd, n_kv_head * head_dim, bias=False)
        self.c_proj = nn.Linear(n_head * head_dim, n_embd,    bias=False)

    def forward(self, x: torch.Tensor,
                cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        H, Hkv, D = self.n_head, self.n_kv_head, self.head_dim

        q = self.c_q(x).view(B, T, H,   D)
        k = self.c_k(x).view(B, T, Hkv, D)
        v = self.c_v(x).view(B, T, Hkv, D)

        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        # QK-normalization  (critical stabilizer in nanochat)
        q = rms_norm(q)
        k = rms_norm(k)

        if self.n_rep > 1:
            k = k.unsqueeze(3).expand(B, T, Hkv, self.n_rep, D).reshape(B, T, H, D)
            v = v.unsqueeze(3).expand(B, T, Hkv, self.n_rep, D).reshape(B, T, H, D)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Cast to bf16: rms_norm promotes to fp32 on XLA; without this
        # the [B,H,T,T] attention score matrix is fp32 → OOM at 2^18 batch.
        _dt = x.dtype
        y = F.scaled_dot_product_attention(
            q.to(_dt), k.to(_dt), v.to(_dt), is_causal=True)

        # ▲ MFU FIX: reshape() instead of the original contiguous().view().
        #   .contiguous() on XLA emits an explicit HBM copy before the
        #   view; .reshape() lets XLA fold the layout change into the
        #   downstream c_proj matmul, eliminating one HBM round-trip per
        #   attention layer per step.
        y = y.transpose(1, 2).reshape(B, T, H * D)
        return self.c_proj(y)


# ── Feed-forward: Squared ReLU (nanochat) ────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, n_embd: int):
        super().__init__()
        self.c_fc   = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.c_proj(F.relu(self.c_fc(x)).square())   # ReLU²


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, head_dim):
        super().__init__()
        self.attn = CausalSelfAttention(n_embd, n_head, n_kv_head, head_dim)
        self.mlp  = MLP(n_embd)

    def forward(self, x, cos, sin):
        h = rms_norm(x)
        return self.attn(h, cos, sin) + self.mlp(h)


# ── Full GPT Model ────────────────────────────────────────────────────────────
class GPT(nn.Module):
    """nanochat GPT — XLA-optimised, no Flash Attention dependency."""

    def __init__(
        self,
        n_layer:    int = N_LAYER,
        n_embd:     int = N_EMBD,
        n_head:     int = N_HEAD,
        n_kv_head:  int = N_KV_HEAD,
        head_dim:   int = HEAD_DIM,
        seq_len:    int = SEQ_LEN,
        vocab_size: int = VOCAB_SIZE,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_embd     = n_embd
        self.n_layer    = n_layer

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, n_kv_head, head_dim) for _ in range(n_layer)]
        )

        # Per-layer residual scalars  (nanochat core feature)
        self.resid_lambdas = nn.Parameter(torch.ones(n_layer))
        self.x0_lambdas    = nn.Parameter(torch.full((n_layer,), 0.1))

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        rc, rs = precompute_rotary(seq_len, head_dim)
        self.register_buffer('rot_cos', rc, persistent=False)
        self.register_buffer('rot_sin', rs, persistent=False)

        self._init_weights()

    def _init_weights(self):
        s = (3 ** 0.5) * (self.n_embd ** -0.5)
        nn.init.normal_(self.wte.weight,      std=1.0)
        nn.init.normal_(self.lm_head.weight,  std=0.001)
        for blk in self.blocks:
            for proj in (blk.attn.c_q, blk.attn.c_k, blk.attn.c_v):
                nn.init.uniform_(proj.weight, -s, s)
            nn.init.zeros_(blk.attn.c_proj.weight)
            nn.init.uniform_(blk.mlp.c_fc.weight, -s, s)
            nn.init.zeros_(blk.mlp.c_proj.weight)
        self.lm_head.weight.data = self.lm_head.weight.data.to(torch.bfloat16)

    def forward(
        self,
        tokens:  torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        B, T = tokens.shape

        x  = self.wte(tokens).to(self.wte.weight.dtype)
        x  = rms_norm(x)
        x0 = x

        cos = self.rot_cos[:, :T]
        sin = self.rot_sin[:, :T]

        for i, blk in enumerate(self.blocks):
            delta = blk(x, cos, sin)
            x = self.resid_lambdas[i] * x + self.x0_lambdas[i] * x0 + delta

        x = rms_norm(x)

        if targets is None:
            logits = self.lm_head(x)
            logits = 30.0 * torch.tanh(logits / 30.0)
            return logits.float()

        # T-chunked CE: splits T into CE_CHUNKS sequential slices so the
        # logit tensor is [B, T//CE_CHUNKS, V] not [B, T, V].
        # At 2^18 batch: [256,1024,50304] bf16 = 26 GB — instant OOM.
        # With CE_CHUNKS=8: [256,128,50304] bf16 = 3.3 GB, processed one
        # chunk at a time so only one lives in HBM simultaneously.
        # XLA unrolls the 8-iter Python loop statically — no graph overhead.
        chunk = x.shape[1] // CE_CHUNKS
        loss  = torch.zeros((), dtype=torch.float32, device=x.device)
        for i in range(CE_CHUNKS):
            x_c = x[:, i * chunk : (i + 1) * chunk]
            t_c = targets[:, i * chunk : (i + 1) * chunk]
            lg  = self.lm_head(x_c)
            lg  = 30.0 * torch.tanh(lg / 30.0)
            loss = loss + F.cross_entropy(
                lg.view(-1, self.vocab_size), t_c.reshape(-1))
        return loss / CE_CHUNKS

    def num_params(self) -> dict:
        wte_p  = self.wte.weight.numel()
        head_p = self.lm_head.weight.numel()
        sc_p   = self.resid_lambdas.numel() + self.x0_lambdas.numel()
        mat_p  = sum(p.numel() for blk in self.blocks for p in blk.parameters())
        total  = wte_p + head_p + sc_p + mat_p
        return dict(wte=wte_p, lm_head=head_p,
                    transformer_matrices=mat_p, scalars=sc_p, total=total)

    def estimate_flops_per_token(self) -> int:
        p = self.num_params()
        return 6 * (p['transformer_matrices'] + p['wte'])


# ── Quick sanity-check ────────────────────────────────────────────────────────
_model_cpu = GPT()
_p         = _model_cpu.num_params()
print('\nParameter breakdown:')
for k, v in _p.items():
    print(f'  {k:<25s}: {v/1e6:7.2f} M')
print(f'\n  FLOPs / token (6N): {_model_cpu.estimate_flops_per_token()/1e9:.3f} GFLOPs')
del _model_cpu



Parameter breakdown:
  wte                      :   38.63 M
  lm_head                  :   38.63 M
  transformer_matrices     :   84.93 M
  scalars                  :    0.00 M
  total                    :  162.20 M

  FLOPs / token (6N): 0.741 GFLOPs


In [10]:
@torch.no_grad()
def zeropower_via_newtonschulz5(G: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """
    G can be 2-D (single matrix) or 3-D (batch of matrices: [N, m, n]).
    Running on a stacked tensor is the key to high MFU on TPU —
    3 big matmuls instead of 72 small ones.
    """
    assert G.ndim >= 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.to(dtype=torch.bfloat16)
    # Work on 'fat' matrices (more cols than rows)
    transposed = X.shape[-2] > X.shape[-1]
    if transposed:
        X = X.mT
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        A = X @ X.mT
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    if transposed:
        X = X.mT
    return X.to(G.dtype)


class Muon(torch.optim.Optimizer):
    """
    Stacked Muon — groups params by shape, runs Newton-Schulz once per
    shape bucket on a [N, m, n] stacked tensor. On TPU v5e this means
    3 big matmuls instead of 72 small ones → ~3-4x better MFU.
    """
    def __init__(self, params, lr=MATRIX_LR, momentum=MUON_MOMENTUM,
                 ns_steps=MUON_NS_STEPS, weight_decay=WEIGHT_DECAY):
        defaults = dict(lr=lr, momentum=momentum,
                        ns_steps=ns_steps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr       = group['lr']
            momentum = group['momentum']
            ns_steps = group['ns_steps']
            wd       = group['weight_decay']

            params = [p for p in group['params'] if p.grad is not None]
            if not params:
                continue

            # ── Group by shape for batched NS ────────────────────────────
            from collections import defaultdict
            shape_groups = defaultdict(list)
            for p in params:
                shape_groups[p.shape].append(p)

            for shape, ps in shape_groups.items():
                # Stack grads: [N, m, n]
                grads = torch.stack([p.grad for p in ps])

                # Nesterov momentum (per-param state, stacked for speed)
                bufs = []
                for p in ps:
                    state = self.state[p]
                    if 'buf' not in state:
                        state['buf'] = torch.zeros_like(p)
                    bufs.append(state['buf'])
                buf_stack = torch.stack(bufs)             # [N, m, n]
                buf_stack.mul_(momentum).add_(grads, alpha=1.0 - momentum)
                g = grads.mul(momentum).add_(buf_stack)  # Nesterov

                # Write momentum buffers back
                for i, p in enumerate(ps):
                    self.state[p]['buf'].copy_(buf_stack[i])

                # Single batched Newton-Schulz call for the whole shape group
                g = zeropower_via_newtonschulz5(g, steps=ns_steps)  # [N, m, n]

                # Scale factor: max(1, m/n)^0.5
                m, n = shape[-2], shape[-1]
                scale = max(1.0, m / n) ** 0.5

                for i, p in enumerate(ps):
                    gi = g[i]
                    if wd > 0:
                        mask = (p.float() * gi.float() > 0).to(p.dtype)
                        p.mul_(1.0 - lr * wd * mask)
                    p.add_(gi, alpha=-lr * scale)


# ── LR schedule (trapezoid with warmup) ──────────────────────────────────────
WARMUP_STEPS   = max(1, int(NUM_STEPS * 0.01))
WARMDOWN_START = int(NUM_STEPS * 0.60)

def get_lr_factor(step: int) -> float:
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    if step < WARMDOWN_START:
        return 1.0
    return max(0.0, 1.0 - (step - WARMDOWN_START) / max(1, NUM_STEPS - WARMDOWN_START))

def update_lr(step, muon, adamw):
    f = get_lr_factor(step)
    for pg in muon.param_groups:
        pg['lr'] = f * MATRIX_LR
    for pg in adamw.param_groups:
        pg['lr'] = f * pg['_base_lr']


def build_optimizers(model: GPT):
    matrix_params = [p for blk in model.blocks
                     for p in blk.parameters() if p.ndim >= 2]
    muon  = Muon(matrix_params, lr=MATRIX_LR, momentum=MUON_MOMENTUM,
                 ns_steps=MUON_NS_STEPS, weight_decay=WEIGHT_DECAY)
    adam_groups = [
        {'params': list(model.wte.parameters()),     '_base_lr': EMBED_LR,  'lr': EMBED_LR},
        {'params': list(model.lm_head.parameters()), '_base_lr': LMHEAD_LR, 'lr': LMHEAD_LR},
        {'params': [model.resid_lambdas],            '_base_lr': SCALAR_LR, 'lr': SCALAR_LR},
        {'params': [model.x0_lambdas],               '_base_lr': X0_LR,     'lr': X0_LR},
    ]
    adamw = torch.optim.AdamW(adam_groups, betas=ADAM_BETAS, eps=ADAM_EPS, weight_decay=0.0)
    print(f'Muon  params: {sum(p.numel() for p in matrix_params)/1e6:.1f} M')
    print(f'AdamW params: {sum(p.numel() for g in adam_groups for p in g["params"])/1e6:.1f} M')
    return muon, adamw

print('Stacked Muon ready.')

Stacked Muon ready.


In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Data Pipeline  — ClimbMix + tiktoken, streamed              ║
# ╚══════════════════════════════════════════════════════════════╝
# Uses tiktoken r50k_base (GPT-2 tokenizer, vocab=50 257).
# Streams ClimbMix from HuggingFace, packs tokens into
# contiguous (B, T+1) slices, no padding needed.
#
# ▲ MFU FIX #4: bulk tokenization with list.extend() instead of
#   per-token yield + append.  The original per-token Python loop
#   ran at ~1–3 M tok/s; extend() runs at ~20 M tok/s, ensuring
#   the prefetch queue is always full and TPU never stalls on data.

import tiktoken
from datasets import load_dataset
import threading, queue
from typing import Iterator

ENC = tiktoken.get_encoding('r50k_base')   # GPT-2 tokenizer, same as ClimbMix
EOT = ENC.eot_token
print(f'Tokenizer: r50k_base  vocab={ENC.n_vocab}  EOT={EOT}')

VAL_TOKENS = 10_000_000


def build_val_set() -> torch.Tensor:
    """Build validation set by consuming the first VAL_TOKENS tokens."""
    print(f'Building val set ({VAL_TOKENS/1e6:.0f}M tokens from ClimbMix)…', flush=True)
    ds = load_dataset(
        'OptimalScale/ClimbMix', split='train',
        streaming=True, trust_remote_code=True,
    )
    buf: list[int] = []
    for example in ds:
        # ▲ bulk extend — avoids per-token Python overhead
        buf.extend(ENC.encode_ordinary(example['text']))
        buf.append(EOT)
        if len(buf) >= VAL_TOKENS:
            break
    t = torch.tensor(buf[:VAL_TOKENS], dtype=torch.int32)
    print(f'Val set: {t.numel():,} tokens')
    return t


class TokenBatcher:
    """
    Background thread streams ClimbMix → tokenises → packs (B, T+1) batches.

    ▲ MFU FIX #4 (cont.): _fill uses list.extend() for bulk tokenisation
      and a document-level skip (not per-token) to skip VAL_TOKENS quickly.
    ▲ MFU FIX #5: prefetch=8 (was 4) — doubles the CPU-side buffer so
      the XLA device always has a fresh batch without blocking.
    """
    def __init__(self, total_seqs: int = TOTAL_SEQS, seq_len: int = SEQ_LEN,
                 skip_first: int = VAL_TOKENS, prefetch: int = 8):
        self.total_seqs = total_seqs
        self.seq_len    = seq_len
        self.n          = total_seqs * (seq_len + 1)
        self._q: queue.Queue = queue.Queue(maxsize=prefetch)
        threading.Thread(
            target=self._fill, kwargs=dict(skip=skip_first), daemon=True,
        ).start()

    def _fill(self, skip: int) -> None:
        ds = load_dataset(
            'OptimalScale/ClimbMix', split='train',
            streaming=True, trust_remote_code=True,
        )
        buf: list[int] = []
        remaining_skip = skip

        for example in ds:
            toks: list[int] = ENC.encode_ordinary(example['text']) + [EOT]

            # ── Fast document-level skip for VAL_TOKENS ───────────────────
            if remaining_skip > 0:
                if remaining_skip >= len(toks):
                    remaining_skip -= len(toks)
                    continue
                # Partial skip: trim the front of this document
                toks = toks[remaining_skip:]
                remaining_skip = 0

            # ── Bulk append — the key throughput improvement ───────────────
            buf.extend(toks)

            # ── Emit complete batches ──────────────────────────────────────
            while len(buf) >= self.n:
                chunk = buf[:self.n]
                buf   = buf[self.n:]
                t = torch.tensor(chunk, dtype=torch.int32).view(
                    self.total_seqs, self.seq_len + 1)
                self._q.put((t[:, :-1].long(), t[:, 1:].long()))

    def __next__(self):
        return self._q.get()

    def __iter__(self):
        return self


print('ClimbMix data pipeline ready (bulk-extend, prefetch=8).')


Tokenizer: r50k_base  vocab=50257  EOT=50256
ClimbMix data pipeline ready (bulk-extend, prefetch=8).


In [12]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SPMD sharding helpers, validation BPB, prompt evaluation    ║
# ╚══════════════════════════════════════════════════════════════╝

def shard_batch(
    x: torch.Tensor,   # (B, T)  on CPU
    y: torch.Tensor,   # (B, T)  on CPU
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Move batch to XLA and annotate the batch dimension as data-parallel.
    Each of the 8 chips will receive 1/8 of the rows automatically.
    """
    x = x.to(DEVICE)
    y = y.to(DEVICE)
    xs.mark_sharding(x, MESH, ('data', None))
    xs.mark_sharding(y, MESH, ('data', None))
    return x, y


@torch.no_grad()
def compute_val_bpb(
    model:    GPT,
    val_data: torch.Tensor,   # flat int32 tensor on CPU
    n_seqs:   int = 128,      # sequences to average over (quick estimate)
    seq_len:  int = SEQ_LEN,
) -> float:
    """
    Bits-per-byte on val_data.  Like nanochat's loss_eval.py —
    vocab-size-independent metric (useful across different tokenizers).

    bpb = (cross_entropy_nats / log(2)) / avg_bytes_per_token
    For tiktoken r50k, avg_bytes_per_token ≈ 4.0.
    """
    BYTES_PER_TOKEN = 4.0   # rough for GPT-2 BPE on English

    model.eval()
    stride  = n_seqs * (seq_len + 1)
    data    = val_data[:stride]

    chunk = data.view(n_seqs, seq_len + 1).long()
    xv, yv = chunk[:, :-1], chunk[:, 1:]
    xv, yv = shard_batch(xv, yv)

    loss = model(xv, yv)
    xm.mark_step()   # flush XLA graph and get scalar
    loss_val = loss.item()
    bpb = (loss_val / math.log(2)) / BYTES_PER_TOKEN

    model.train()
    return bpb


def grad_norm(model: GPT) -> float:
    """Global gradient L2 norm (for monitoring)."""
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.detach().float().norm().item() ** 2
    return total ** 0.5


# ── Prompt evaluation (runs on CPU to avoid per-token XLA sync) ──────────────
@torch.no_grad()
def generate_samples(
    model:       GPT,
    prompts:     list[str] = EVAL_PROMPTS,
    max_new:     int        = 100,
    temperature: float      = 0.8,
    top_k:       int        = 50,
    step:        int        = 0,
) -> list[str]:
    """
    Evaluate qualitative generation on EVAL_PROMPTS every SAMPLE_EVAL_EVERY steps.

    Strategy: copy model weights to a temporary CPU model (fp32).
      • Avoids the per-token XLA mark_step() overhead that would otherwise
        pause the TPU for each of (len(prompts) × max_new) token decisions.
      • The CPU copy is ~650 MB and takes ~0.3 s to copy; this is negligible
        at the 3 000-step cadence (called roughly every 30 min of training).
      • rot_cos / rot_sin are non-persistent buffers — recreated in GPT.__init__.
    """
    # ── Build a temporary CPU model ───────────────────────────────────────────
    # Copy all weights to CPU as fp32.
    # lm_head.weight is stored as bf16 (set in _init_weights); without the
    # explicit .float() cast here the CPU matmul would mix bf16 weights with
    # fp32 activations and raise "expected m1 and m2 to have the same dtype".
    cpu_sd = {k: v.cpu().float() for k, v in model.state_dict().items()}
    cpu_model = GPT()                  # rot buffers recreated automatically
    cpu_model.load_state_dict(cpu_sd, strict=True)
    cpu_model.float()                  # ensure ALL buffers/params are fp32
    cpu_model.eval()
    del cpu_sd

    results: list[str] = []

    sep = '═' * 72
    print(f'\n{sep}')
    print(f'  PROMPT EVAL  @  step {step:,}  ({len(prompts)} prompts, max_new={max_new})')
    print(sep)

    for prompt in prompts:
        ids = torch.tensor([ENC.encode_ordinary(prompt)], dtype=torch.long)

        for _ in range(max_new):
            ids_cond = ids[:, -SEQ_LEN:]
            # fp32 inference on CPU — small model, still fast
            logits = cpu_model(ids_cond)[:, -1, :].float() / temperature
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs  = torch.softmax(logits, dim=-1)
            nxt    = torch.multinomial(probs, num_samples=1)
            if nxt.item() == EOT:
                break
            ids = torch.cat([ids, nxt], dim=1)

        text = ENC.decode(ids[0].tolist())
        results.append(text)
        # Print only the generated continuation (after the prompt)
        continuation = text[len(prompt):]
        print(f'\n  PROMPT : {prompt}')
        print(f'  OUTPUT : {prompt}\033[1m{continuation}\033[0m')

    print(f'\n{sep}\n')
    del cpu_model   # free CPU RAM immediately

    # ── Optional wandb logging ────────────────────────────────────────────────
    if USE_WANDB:
        import wandb
        rows = [[p, r[len(p):]] for p, r in zip(prompts, results)]
        table = wandb.Table(columns=["prompt", "continuation"], data=rows)
        wandb.log({"generated_samples": table}, step=step)

    model.train()   # restore training mode
    return results


print('SPMD helpers + generate_samples() ready.')


SPMD helpers + generate_samples() ready.


In [13]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Main Training Loop                                          ║
# ╚══════════════════════════════════════════════════════════════╝

import torch_xla

def pretrain(
    resume_step: int  = 0,    # set > 0 to resume from checkpoint
    max_steps:   int  = NUM_STEPS,
):
    # ── Build model on XLA ─────────────────────────────────────────────────
    model = GPT().to(DEVICE)
    xm.mark_step()   # let XLA materialise the empty parameter tensors

    p = model.num_params()
    flops_pt = model.estimate_flops_per_token()
    print(f'Model total   : {p["total"]/1e6:.1f} M params')
    print(f'  — wte       : {p["wte"]/1e6:.1f} M')
    print(f'  — lm_head   : {p["lm_head"]/1e6:.1f} M')
    print(f'  — matrices  : {p["transformer_matrices"]/1e6:.1f} M  ← N for scaling law')
    print(f'FLOPs/token  : {flops_pt/1e9:.3f} G  (6N approx)')
    print(f'Total train FLOPs: ~{flops_pt*TARGET_TOKENS:.3e}')

    # ── Optimizers ────────────────────────────────────────────────────────
    muon, adamw = build_optimizers(model)

    # ── Optionally resume ─────────────────────────────────────────────────
    start_step = 0
    if resume_step > 0:
        ckpt_path = f'{CKPT_DIR}/step_{resume_step:06d}.pt'
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location='cpu')
            model.load_state_dict(ckpt['model'], strict=True)
            muon.load_state_dict(ckpt['muon'])
            adamw.load_state_dict(ckpt['adamw'])
            start_step = resume_step
            print(f'Resumed from step {start_step}')
        else:
            print(f'WARNING: checkpoint {ckpt_path} not found, starting fresh.')

    # ── Data ──────────────────────────────────────────────────────────────
    print('\nBuilding validation set …')
    val_data = build_val_set()       # 10 M tokens, CPU

    print('\nStarting training data stream …')
    batcher = TokenBatcher(skip_first=VAL_TOKENS)  # skip val tokens

    # ── wandb (optional) ──────────────────────────────────────────────────
    if USE_WANDB:
        import wandb
        wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN,
            config=dict(
                depth=DEPTH, n_embd=N_EMBD, n_head=N_HEAD,
                seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE,
                total_tokens=TARGET_TOKENS, batch_tokens=TOTAL_BATCH_TOKENS,
                total_steps=NUM_STEPS, num_devices=NUM_DEVICES,
                matrix_lr=MATRIX_LR, muon_momentum=MUON_MOMENTUM,
                weight_decay=WEIGHT_DECAY, warmdown_start=WARMDOWN_START,
                sample_eval_every=SAMPLE_EVAL_EVERY,
            ),
        )

    # ── Training ──────────────────────────────────────────────────────────
    model.train()
    t0 = time.perf_counter()
    step0_time = t0

    print(f'\n{"Step":>6}  {"loss":>8}  {"val_bpb":>9}  '
          f'{"tok/s":>9}  {"MFU%":>6}  lr_scale')
    print('-' * 65)

    # TPU peak TFLOPs/chip for v5e (bf16 matmuls)
    TPU_V5E_TFLOPS_PER_CHIP = 197.0   # bf16, single-chip peak
    PEAK_TFLOPS = TPU_V5E_TFLOPS_PER_CHIP * NUM_DEVICES * 1e12

    for step in range(start_step, max_steps):

        # ── LR schedule ──────────────────────────────────────────────────
        update_lr(step, muon, adamw)
        lr_f = get_lr_factor(step)

        for p in model.parameters():
            p.grad = None

        # ── Get next batch ────────────────────────────────────────────────
        x_cpu, y_cpu = next(batcher)    # (TOTAL_SEQS, SEQ_LEN)  int64 CPU
        x, y = shard_batch(x_cpu, y_cpu)

        with torch.autocast('xla', dtype=torch.bfloat16):
            loss = model(x, y)

        loss.backward()

        # ── Optimizer step + XLA graph commit ─────────────────────────
        muon.step()
        adamw.step()
        xm.mark_step()

        # ── Logging ───────────────────────────────────────────────────
        if step % LOG_EVERY == 0:
            t1 = time.perf_counter()
            dt = t1 - t0
            t0 = t1

            loss_val = loss.item()    # implicitly syncs TPU

            # Throughput
            toks_per_sec = TOTAL_BATCH_TOKENS * LOG_EVERY / dt
            # MFU = actual_flops / theoretical_peak_flops
            actual_tflops = toks_per_sec * flops_pt / 1e12
            mfu = 100.0 * actual_tflops / (PEAK_TFLOPS / 1e12)

            # Tokens seen so far
            tokens_seen = (step + 1) * TOTAL_BATCH_TOKENS

            print(f'{step:>6d}  {loss_val:>8.4f}  '
                  f'{"":>9}  '
                  f'{toks_per_sec/1e6:>7.3f}M  '
                  f'{mfu:>5.1f}%  '
                  f'{lr_f:.3f}')

            if USE_WANDB:
                import wandb
                wandb.log(dict(
                    step=step, loss=loss_val,
                    tok_per_sec=toks_per_sec,
                    mfu=mfu, lr_factor=lr_f,
                    tokens_seen=tokens_seen,
                ), step=step)

        # ── Validation ────────────────────────────────────────────────
        if step % VAL_EVERY == 0 and step > 0:
            bpb = compute_val_bpb(model, val_data)
            print(f'{step:>6d}  {"":>8}  {bpb:>9.5f}  '
                  f'{"":>9}  {"":>6}  {"val"}')
            if USE_WANDB:
                import wandb
                wandb.log({'val_bpb': bpb, 'step': step}, step=step)

        # ── Prompt Evaluation ─────────────────────────────────────────
        # NEW: every SAMPLE_EVAL_EVERY steps, generate text from EVAL_PROMPTS.
        # Uses a temporary CPU copy of the model weights so the XLA device
        # is NOT involved in token-by-token generation (no per-token syncs).
        if step % SAMPLE_EVAL_EVERY == 0 and step > 0:
            generate_samples(model, step=step)

        # ── Checkpoint ────────────────────────────────────────────────
        if step % CKPT_EVERY == 0 and step > 0:
            ckpt_path = f'{CKPT_DIR}/step_{step:06d}.pt'
            # Pull weights to CPU before saving
            cpu_sd = {k: v.cpu() for k, v in model.state_dict().items()}
            torch.save({
                'step':  step,
                'model': cpu_sd,
                'muon':  muon.state_dict(),
                'adamw': adamw.state_dict(),
                'config': dict(
                    n_layer=N_LAYER, n_embd=N_EMBD, n_head=N_HEAD,
                    n_kv_head=N_KV_HEAD, head_dim=HEAD_DIM,
                    seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE,
                ),
            }, ckpt_path)
            print(f'  [ckpt] saved {ckpt_path}')

    # ── Final validation ──────────────────────────────────────────────────
    final_bpb = compute_val_bpb(model, val_data, n_seqs=512)
    total_time = time.perf_counter() - step0_time
    print(f'\n═══ Training complete ═══')
    print(f'Total time   : {total_time/3600:.2f} h')
    print(f'Final val bpb: {final_bpb:.5f}')
    print(f'Tokens trained: {max_steps*TOTAL_BATCH_TOKENS/1e9:.2f} B')

    # ── Final prompt evaluation ────────────────────────────────────────────
    print('\nFinal prompt evaluation:')
    generate_samples(model, step=max_steps, max_new=150)

    # Final checkpoint
    cpu_sd = {k: v.cpu() for k, v in model.state_dict().items()}
    torch.save({'step': max_steps, 'model': cpu_sd,
                'muon': muon.state_dict(), 'adamw': adamw.state_dict()},
               f'{CKPT_DIR}/final.pt')
    print(f'Final checkpoint: {CKPT_DIR}/final.pt')

    if USE_WANDB:
        import wandb
        wandb.finish()

    return model


print('pretrain() defined.  Next cell: launch training.')


pretrain() defined.  Next cell: launch training.


In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  LAUNCH  — runs full Chinchilla-optimal pretraining          ║
# ╚══════════════════════════════════════════════════════════════╝
#
# Estimated wall-clock on TPU v5e-8:
#   ~25-40 min depending on data-fetch latency and XLA compilation.
#   The first few steps are slow (XLA JIT tracing); steady-state
#   throughput kicks in after ~50 steps.
#
# To resume a run:
#   model = pretrain(resume_step=1000)

DEVICE = torch.device('xla')

model = pretrain()

/tmp/ipykernel_12/3415581763.py:13: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()   # let XLA materialise the empty parameter tensors


Model total   : 162.2 M params
  — wte       : 38.6 M
  — lm_head   : 38.6 M
  — matrices  : 84.9 M  ← N for scaling law
FLOPs/token  : 0.741 G  (6N approx)
Total train FLOPs: ~1.839e+18
Muon  params: 84.9 M
AdamW params: 77.3 M

Building validation set …
Building val set (10M tokens from ClimbMix)…


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'OptimalScale/ClimbMix' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'OptimalScale/ClimbMix' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


     0   10.8261               0.114M    5.4%  0.005
    50    5.7654               0.243M   11.4%  0.270
   100    5.0871               0.493M   23.2%  0.534
   150    5.1212               0.265M   12.5%  0.799
   200    4.6745               0.492M   23.1%  1.000


/tmp/ipykernel_12/1746839350.py:45: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()   # flush XLA graph and get scalar


   200              1.66200                     val
   250    4.6909               0.384M   18.1%  1.000
   300    4.1305               0.492M   23.2%  1.000
   350    3.8926               0.252M   11.9%  1.000
   400    4.1825               0.493M   23.2%  1.000
   400              1.45693                     val
   450    4.1146               0.486M   22.9%  1.000
   500    3.8079               0.492M   23.2%  1.000
   550    3.6044               0.491M   23.1%  1.000
   600    3.7722               0.492M   23.1%  1.000
   600              1.33623                     val
   650    3.5965               0.486M   22.9%  1.000
   700    2.9949               0.493M   23.2%  1.000
   750    2.9520               0.492M   23.1%  1.000
   800    3.2152               0.492M   23.1%  1.000
   800              1.31271                     val
   850    2.9806               0.488M   22.9%  1.000
   900    3.4807               0.492M   23.2%  1.000
   950    3.2665               0.492M   23.1%  1.0

RuntimeError: expected m1 and m2 to have the same dtype, but got: c10::BFloat16 != float

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Sampling (greedy / top-k / temperature)                     ║
# ╚══════════════════════════════════════════════════════════════╝
# NOTE: generate_samples() (defined in the SPMD helpers cell) is used
# during training. This cell provides a convenient interactive wrapper.

@torch.no_grad()
def sample(
    model:       GPT,
    prompt:      str,
    max_new:     int   = 200,
    temperature: float = 0.8,
    top_k:       int   = 50,
) -> str:
    """
    Interactive sampling.  Runs on CPU to avoid per-token XLA sync overhead
    (each token would require a mark_step() on the TPU device otherwise).
    """
    cpu_sd = {k: v.cpu().float() for k, v in model.state_dict().items()}
    cpu_model = GPT()
    cpu_model.load_state_dict(cpu_sd, strict=True)
    cpu_model.float()   # lm_head.weight is bf16 in state_dict; cast to fp32
    cpu_model.eval()

    ids = torch.tensor([ENC.encode_ordinary(prompt)], dtype=torch.long)
    for _ in range(max_new):
        ids_cond = ids[:, -SEQ_LEN:]
        logits   = cpu_model(ids_cond)[:, -1, :].float() / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
        probs   = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        if next_id.item() == EOT:
            break
        ids = torch.cat([ids, next_id], dim=1)

    del cpu_model
    return ENC.decode(ids[0].tolist())


# Test generation after training
prompts = [
    'The capital of France is',
    'In 1969, humans first',
    'The chemical formula for water is',
    'Machine learning is a field of',
]

print('=== Samples from trained model ===\n')
for p in prompts:
    out = sample(model, p, max_new=80, temperature=0.8)
    print(f'>>> {p}')
    print(out)
    print()


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Load & evaluate any saved checkpoint                        ║
# ╚══════════════════════════════════════════════════════════════╝

def load_checkpoint(path: str) -> GPT:
    ckpt  = torch.load(path, map_location='cpu')
    cfg   = ckpt.get('config', {})
    m     = GPT(**{k: v for k, v in cfg.items() if k in GPT.__init__.__code__.co_varnames})
    m.load_state_dict(ckpt['model'], strict=True)
    m     = m.to(DEVICE)
    xm.mark_step()
    print(f'Loaded checkpoint from step {ckpt.get("step", "?")}  →  {path}')
    return m


# Example usage:
#   m = load_checkpoint('./checkpoints/final.pt')
#   bpb = compute_val_bpb(m, val_data, n_seqs=512)
#   print(f'val bpb = {bpb:.5f}')

print('load_checkpoint() defined.')
print('Run: m = load_checkpoint("./checkpoints/final.pt")')


## Notes, Tuning Tips & Next Steps

### Architecture features from nanochat (all included)
| Feature | Source | Effect |
|---------|--------|--------|
| RoPE rotary embeddings | nanochat | Better length generalisation |
| QK-normalization | nanochat | Training stability, prevents attn entropy collapse |
| ReLU² activation | nanochat/modded-nanogpt | Cleaner sparsity than GELU |
| Per-layer λ_resid / λ_x0 scalars | nanochat | +0.003–0.01 bpb improvement |
| Logit soft-cap 30·tanh(·/30) | nanochat | Prevents logit blow-up |
| Untied wte / lm_head | nanochat | Independent embedding optimisation |
| Zero-init projections | nanochat | Better early-training dynamics |
| Muon + AdamW split | nanochat | Muon ≫ AdamW for matrices |
| Trapezoidal LR (0 warmup, 40% WD) | nanochat | Best schedule for fixed budget |

### Value Embeddings (not included — adds ~231 M extra params at depth-12)
nanochat uses per-layer token-level value embeddings at alternating layers.  
Add them back in for **larger models** (depth ≥ 20) where the extra capacity  
is meaningful without doubling total parameter count.

### Scaling to larger depth
```python
DEPTH = 20  # → n_embd=1280, ~560 M total params, ~10 B Chinchilla tokens
DEPTH = 24  # → n_embd=1536, GPT-2 1.5B-class, ~3 hr on v5e-8
```

### GCS checkpointing
```bash
export CKPT_DIR = 'gs://your-bucket/nanochat-runs/d12'
```
torch.save / torch.load work transparently with gs:// paths via gcsfs.

### FP8 training (TPU v5e supports it)
nanochat uses FP8 via transformer-engine on H100. For TPU v5e, XLA's  
built-in BF16 is already ~optimal; FP8 support in torch_xla is experimental.

### SFT after pretraining
Load `./checkpoints/final.pt`, clone nanochat and run:  
```bash
python -m scripts.chat_sft --resume-from ./checkpoints/final.pt
```
